In [2]:
import os
import warnings
from dotenv import load_dotenv
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
warnings.filterwarnings("ignore")
load_dotenv()

True

In [3]:
from langchain_ollama import ChatOllama 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.tools import tool


# llm = ChatOllama(model='llama3.2:3b', base_url='http://localhost:11434')

In [10]:
%pip install pymysql langchain-groq


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from sqlalchemy import create_engine
from langchain_community.utilities import SQLDatabase

db_uri = os.environ.get('DATABASE_URI')
if db_uri is None:
    raise ValueError("DATABASE_URI environment variable not set")
db = SQLDatabase.from_uri(db_uri)


In [5]:
db.dialect
db.get_usable_table_names()
db.run("SELECT * FROM events LIMIT 5")

"[('674c75979ef0a1a71bb4f6e9', 'Marriage and Reception', 'Celebrate your special day with elegance and style! Our expert event organizers will help you host a memorable marriage reception, handling everything from decor to seamless coordination. Whether intimate or grand, we customize every detail to match your vision. Let us make your dream reception...', 'Gopal Maidan', 'https://i.ibb.co/x2RpYrx/original.jpg', datetime.datetime(2024, 12, 2, 14, 35, 40), datetime.datetime(2025, 1, 31, 14, 35, 40), Decimal('1500.00'), 0, 'https://www.youtube.com/embed/eTl2Cxb74r0', '674c707059b10fc9cf1fc465', '674c73e21d1ffda36b808d94', datetime.datetime(2024, 12, 1, 14, 41, 27)), ('674c78c4673f030cec090834', 'Engagement', 'Make your engagement unforgettable with our expert event organizing services!', 'Gandhi maidan', 'https://i.ibb.co/tQCQdzL/eng.webp', datetime.datetime(2024, 12, 1, 14, 49, 33), datetime.datetime(2024, 12, 31, 14, 49, 33), Decimal('120.00'), 0, 'https://www.youtube.com/embed/XbFfVwt

In [6]:
from langchain.chains import create_sql_query_chain
from langchain_groq import ChatGroq

# Convert Groq client to LangChain ChatGroq
llm = ChatGroq(
	groq_api_key=os.getenv("GROQ_API_KEY"),
	model_name="llama-3.3-70b-versatile"
)

sql_chain = create_sql_query_chain(llm, db)

In [7]:
question = "how many events are there? You MUST RETURN ONLY MYSQL QUERIES."
response = sql_chain.invoke({'question': question})
print(response)

SQLQuery: SELECT COUNT(`id`) FROM events LIMIT 1


In [8]:

from langchain_core.prompts import (SystemMessagePromptTemplate, 
                                    HumanMessagePromptTemplate,
                                    ChatPromptTemplate)

In [9]:

system = SystemMessagePromptTemplate.from_template("""You are helpful AI assistant who answer user question based on the provided context.""")

prompt = """Answer user question based on the provided context ONLY! If you do not know the answer, just say "I don't know".
            ### Context:
            {context}

            ### Question:
            {question}

            ### Answer:"""

prompt = HumanMessagePromptTemplate.from_template(prompt)

messages = [system, prompt]
template = ChatPromptTemplate(messages)

qna_chain = template | llm | StrOutputParser()

def ask_llm(context, question):
    return qna_chain.invoke({'context': context, 'question': question})

In [10]:
from langchain_core.runnables import chain
@chain
def get_correct_sql_query(input):
    context = input['context']
    question = input['question']

    intruction = """
        Use above context to fetch the correct SQL query for following question
        {}

        Do not enclose query in ```sql and do not write preamble and explanation.
        You MUST return only single SQL query.
    """.format(question)

    response = ask_llm(context=context, question=intruction)

    return response

In [11]:
response = get_correct_sql_query.invoke({'context': response, 'question': question})
db.run(response)

'[(6,)]'

In [12]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

In [13]:
execute_query = QuerySQLDataBaseTool(db=db)
sql_query = create_sql_query_chain(llm, db)

final_chain = (
    {'context': sql_query, 'question': RunnablePassthrough()}
    | get_correct_sql_query
    | execute_query | StrOutputParser()
)
question = "Number of events price below 100?? You MUST RETURN ONLY MYSQL QUERIES."

response = final_chain.invoke({'question': question})
print(response)

/var/folders/bg/tzv6rcf554g34dtzyp8h9mtw0000gn/T/ipykernel_15848/674384762.py:1: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-community package and should be used instead. To use it run `pip install -U :class:`~langchain-community` and import as `from :class:`~langchain_community.tools import QuerySQLDatabaseTool``.
  execute_query = QuerySQLDataBaseTool(db=db)


[(2,)]


In [14]:
# %pip install --upgrade --quiet langgraph
# %%capture --no-stderr
# %pip install -U tavily-python langchain_community

In [15]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.tools.tavily_search import TavilySearchResults

tool = TavilySearchResults(max_results=2)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()
tools = tools + [tool]

In [ ]:
from langchain_core.messages import SystemMessage

SQL_PREFIX = """You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct SQLite query to run, then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most 10 results.
You can order the results by a relevant column to return the most related examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.
You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.


DO NOT INCLUDE DATABASE CONNECTION INFORMATION IN YOUR QUERY. GIVE HUMAN READABLE ANSWERS.
DO NOT GIVE INFORMATION ABOUT THE DATABASE LIKE TABLE NAME OR COLUMN NAME.


DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.

To start you should ALWAYS look at the tables in the database to see what you can query.
Do NOT skip this step.
Then you should query the schema of the most relevant tables."""

system_message = SystemMessage(content=SQL_PREFIX)

In [17]:
%pip install langgraph
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(llm, tools, state_modifier=system_message, debug=False)


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip3.11 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
question = "Events price below 100??"
# question = "How many departments are there?"

agent_executor.invoke({"messages": [HumanMessage(content=question)]})

{'messages': [HumanMessage(content='Events price below 100??', additional_kwargs={}, response_metadata={}, id='6b308ff1-cabd-4933-9e9c-88305f126936'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_5v0m', 'function': {'arguments': '{"tool_input": ""}', 'name': 'sql_db_list_tables'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 1123, 'total_tokens': 1139, 'completion_time': 0.059394784, 'prompt_time': 0.047332905, 'queue_time': 0.049549527, 'total_time': 0.106727689}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_e91e6fbd56', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-bdd87bce-c366-470e-8dde-7209c4611a32-0', tool_calls=[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': 'call_5v0m', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1123, 'output_tokens': 16, 'total_tokens': 1139}),
  ToolMessage(content='events', name='sql_db_list_tables', id='f687c028-af

In [19]:
for s in agent_executor.stream(
    {"messages": [HumanMessage(content=question)]}
):
    print(s) 
    print("----")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_yf3m', 'function': {'arguments': '{"tool_input": ""}', 'name': 'sql_db_list_tables'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 1123, 'total_tokens': 1139, 'completion_time': 0.058181818, 'prompt_time': 0.05764543, 'queue_time': 0.04901459, 'total_time': 0.115827248}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_2ddfbb0da0', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-74308378-c897-4b91-802f-86971efdbfac-0', tool_calls=[{'name': 'sql_db_list_tables', 'args': {'tool_input': ''}, 'id': 'call_yf3m', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1123, 'output_tokens': 16, 'total_tokens': 1139})]}}
----
{'tools': {'messages': [ToolMessage(content='events', name='sql_db_list_tables', id='da1f2f5e-e7b3-4da0-ae3d-32c18db95e6c', tool_call_id='call_yf3m')]}}
----
{'agent': {'messages': [AIMessage(conten

In [20]:
print(agent_executor.invoke({"messages": [HumanMessage(content=question)]})['messages'][-1].content)

The events with a price below 100 are:
- Birthday Party: $50.00
- Marriage Anniversary: $70.00
All the dosages included in the database for these events are not available as the database does not contain any information about dosages for events.
